> ## SOLUTION / ANSWER KEY &mdash; Lab 5.11
> This is the **completed** notebook (all `___` blanks filled). For the student version, open
> [`../lab-11-reflection-self-critique.ipynb`](../lab-11-reflection-self-critique.ipynb). Trainer use &mdash; or self-check after you've tried it yourself.

# Lab 5.11 &mdash; Reflection: Self-Critique and Revise

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 5 &mdash; What is Agentic AI?**

### What you'll do
- Let the REAL model produce a first (often wrong) numeric answer
- Run a grounded critic that checks it against a DERIVED truth
- Revise with the model when the critic objects, and confirm the fix

> **How this lab works (near-real):** you have a local Ollama running `llama3.1:8b`. In Module 5 you build the agent **from scratch** &mdash; the loop, the ReAct parser, the tool router, the memory, the guardrails &mdash; as **real Python**. What's new vs the old version: a **real local model** now does the *reasoning* (it emits the ReAct steps / picks the actions) while **your** code parses, routes and loops it, and **real tools** run. Read the **Concept**, fill the real `___` blanks in **Build it**, then **Run it for real** and **read the trace**. Finish with an open **Your turn**. There is **no auto-grader**.

> **Framework note:** these labs use a **real local model** (`ollama run llama3.1:8b`, pinned to `http://127.0.0.1:11434`) via `langchain-ollama`. Unlike Module 6, you do **not** hand the loop to a framework &mdash; you build it yourself and the model drives it. If Ollama is down, the run cells print how to start it instead of crashing. A tool must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole loop (you'll see exactly this in Labs 5 and 8). A small model can pick a wrong tool or fumble a step now and then &mdash; that's real agent behaviour, and it's why you read the trace.

**Reference:** [Module 5 slides &mdash; Planning patterns at a glance](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 5 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, socket, pathlib
from dotenv import load_dotenv
load_dotenv(pathlib.Path("/home/rajesh/Training/courses/building-intelligents-ai-agents/.env"))   # GROQ/OPENAI keys, if you ever want a hosted alternative

WORK = "/tmp/biaa-lab-05-11"
os.makedirs(WORK, exist_ok=True)

def ollama_up(host="127.0.0.1", port=11434):
    """True if a local Ollama server is listening. If it's down, start it with:  ollama run llama3.1:8b"""
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

from langchain_ollama import ChatOllama
# Day-3 provider: a REAL local model DRIVES the agent YOU build from scratch.
# Pin the host -- plain 'localhost' can give 'No route to host'.
llm = ChatOllama(model="llama3.1:8b", temperature=0, base_url="http://127.0.0.1:11434")

def llm_text(prompt):
    """Call the REAL model and return its text (the .content of the reply)."""
    return llm.invoke(prompt).content

if ollama_up():
    print("Ollama reachable at 127.0.0.1:11434 | model:", llm.model, "| WORK:", WORK)
else:
    print("Ollama NOT reachable -- start it with:  ollama run llama3.1:8b")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
**Reflection** has the agent **critique its own output** and improve it before answering. Here the **real
model** drafts an answer (a small model often forgets to double the population), a **grounded critic**
&mdash; deterministic, real, derived from the looked-up base &mdash; catches the mistake, and the **real
model revises**. The critic checks a **rule** (*answer == twice the base*), **not** a hardcoded `240000`, so
it would still be right if the base changed. Reflection trades extra calls for higher quality.

In [ ]:
# DEMO -- the task: report TWICE the base population. The truth is DERIVED from the looked-up base,
# so the critic reasons about a rule (2 * base) rather than matching one magic constant.
KB = {"population of metropolis": 120000}
print("base:", KB["population of metropolis"], "-> correct answer is 2 * base =", 2 * KB["population of metropolis"])

## Build it
Implement the derived `critic` target and the `reflect` decision (revise only when the critic **rejected**
the draft). `draft`/`revise` call the real model; `extract_number` reads a number out of its prose.

In [ ]:
import re
KB = {"population of metropolis": 120000}
def base_population():
    return KB["population of metropolis"]

def extract_number(text):
    nums = re.findall(r"-?\d[\d,]*", str(text))
    return int(nums[-1].replace(",", "")) if nums else None

def draft(task):
    """A REAL first attempt -- a small model often forgets to double."""
    return llm_text("Answer with JUST a number, no words. " + task)

def critic(answer_text):
    # DERIVED grounded check: the answer must equal twice the base population (not hardcoded 240000).
    want = 2 * base_population()
    got = extract_number(answer_text)
    if got == want:
        return (True, "correct")
    return (False, "expected " + str(want) + " (twice the base population), got " + str(got))

def revise(task, feedback):
    """Ask the REAL model to fix its answer given the critic's feedback."""
    return llm_text("Your previous answer was wrong: " + feedback + ". " + task +
                    " Reply with JUST the corrected number.")

def reflect(task):
    first = draft(task)
    ok, fb = critic(first)
    final = first if ok else revise(task, fb)
    return {"first": first, "ok_first": ok, "feedback": fb, "final": final}

try:
    print("critic on '240000':", critic("240000"))
    print("critic on '120000':", critic("120000"))   # the un-doubled base -> rejected
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## Run it for real &amp; read the trace
Run the full reflect loop: the real model drafts, your grounded critic checks, the real model revises if needed.

_This calls the real `llama3.1:8b` via Ollama. If Ollama is down the cell prints how to start it instead of crashing._

In [ ]:
if not ollama_up():
    print("Ollama not reachable -- start `ollama run llama3.1:8b`, then re-run this cell.")
else:
    try:
        out = reflect("What is twice the population of metropolis (120000)?")
        print("DRAFT  :", out["first"])
        print("CRITIC :", "approved" if out["ok_first"] else "REJECTED -- " + out["feedback"])
        print("FINAL  :", out["final"])
    except Exception as e:
        print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- The **real** model drafted an answer; if it returned the un-doubled base, the **deterministic** critic caught it and the model **revised**.
- The critic is **derived** (`2 * base_population()`) &mdash; change the base and it's still correct, unlike a hardcoded `== 240000`.
- Reflection = **draft -> critique -> revise**. It costs extra model calls but lifts quality &mdash; reach for it when being right matters more than being fast.

## Your turn (open task &mdash; no grader)
Make the critic itself an **LLM critic**: ask the model *"is this answer twice 120000? reply yes/no"* and
compare to your grounded check. **What good looks like:** the grounded (deterministic) critic is reliable;
the LLM critic is flexible but can be wrong &mdash; a real trade-off you'll weigh when designing agents.

---
### Key takeaway
Reflection = answer, critique, revise. A REAL model drafts and revises; a deterministic grounded critic keeps it honest. Extra calls, higher quality.

[&#8592; All Module 5 labs](./index.html) &nbsp;&middot;&nbsp; [Module 5 slides](../../presentation/day3-module5-what-is-agentic-ai.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>